# FMP D&A / CAPEX / NWC 원천 데이터 확인
## 분기별 실제 수신 데이터 vs 코드 계산값 비교

| 섹션 | 내용 |
|---|---|
| Cell 1 | 설정 (Ticker / API Key / 연도) |
| Cell 2 | FMP fetch 함수 & 날짜 파싱 |
| Cell 3 | 데이터 수신 (CF + BS) |
| Cell 4 | **[A]** CF Statement 전체 컬럼 확인 |
| Cell 5 | **[B]** D&A·CAPEX·Working Capital 관련 컬럼 원본값 |
| Cell 6 | **[C]** 핵심 항목 요약 — CF Statement + BS NWC 나란히 |
| Cell 7 | **[D]** CAPEX 부호 검증 (`take_abs=True` 효과) |
| Cell 8 | **[E]** 최근 12분기 이력 + α·β·γ 비율 |
| Cell 9 | **[F]** 계수 중앙값 요약 |

In [1]:
import requests
import pandas as pd
import time

# ════════════════════════════════════════════════════════════
#  ★ 설정 — 여기만 수정하세요 ★
# ════════════════════════════════════════════════════════════
TICKER       = "NFLX"                          # 분석 티커
TARGET_YEAR  = 2025                            # 확인 연도
FMP_API_KEY  = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"
FMP_BASE     = "https://financialmodelingprep.com/api/v3"
LIMIT        = 20                              # 최근 N분기
SLEEP        = 0.35

pd.set_option("display.float_format", "{:,.0f}".format)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 50)

print(f"[설정] Ticker={TICKER}  Target={TARGET_YEAR}  Limit={LIMIT}분기")

[설정] Ticker=NFLX  Target=2025  Limit=20분기


In [2]:
# ════════════════════════════════════════════════════════════
#  FMP fetch 함수 & 날짜 파싱
# ════════════════════════════════════════════════════════════

def fmp_get(endpoint: str, ticker: str, limit: int = LIMIT) -> pd.DataFrame:
    """FMP API 분기별 재무제표 수신 (3회 retry)"""
    url = f"{FMP_BASE}/{endpoint}/{ticker}"
    for attempt in range(3):
        try:
            r = requests.get(url,
                             params={"period": "quarter",
                                     "limit": limit,
                                     "apikey": FMP_API_KEY},
                             timeout=20)
            if r.status_code == 429:
                time.sleep(2 + attempt)
                continue
            r.raise_for_status()
            data = r.json()
            if isinstance(data, dict) and "Error Message" in data:
                print(f"  [FMP Error] {data['Error Message']}")
                return pd.DataFrame()
            return pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame()
        except Exception as e:
            if attempt == 2:
                print(f"  [FAIL] {endpoint}: {e}")
                return pd.DataFrame()
            time.sleep(SLEEP + attempt * 0.5)
    return pd.DataFrame()


def parse_qend(df: pd.DataFrame) -> pd.DataFrame:
    """
    acceptedDate - 45일 → 분기말 매핑
    Company_Investment_Report 의 parse_qend 와 동일 로직
    """
    df = df.copy()
    df["report_date"] = pd.to_datetime(
        df.get("acceptedDate", df.get("fillingDate", pd.NaT)),
        errors="coerce"
    )
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    def _qend(row):
        if pd.notna(row.get("report_date")):
            return (row["report_date"] - pd.Timedelta(days=45)) \
                       .to_period("Q").to_timestamp("Q")
        return row["date"].to_period("Q").to_timestamp("Q")

    df["date_mapped"] = df.apply(_qend, axis=1)
    return df.sort_values("date_mapped", ascending=False).reset_index(drop=True)


print("[OK] FMP fetch 함수 & parse_qend 정의 완료")

[OK] FMP fetch 함수 & parse_qend 정의 완료


In [5]:
# ════════════════════════════════════════════════════════════
#  데이터 수신 — CF Statement + Balance Sheet
# ════════════════════════════════════════════════════════════

print(f"[1/2] Cash Flow Statement 수신 중...  ({TICKER})")
cf_raw = fmp_get("cash-flow-statement", TICKER)
time.sleep(SLEEP)

print(f"[2/2] Balance Sheet Statement 수신 중...  ({TICKER})")
bs_raw = fmp_get("balance-sheet-statement", TICKER)

if cf_raw.empty:
    raise ValueError("[ERROR] Cash Flow Statement 수신 실패 — API Key 또는 티커를 확인하세요")
if bs_raw.empty:
    raise ValueError("[ERROR] Balance Sheet Statement 수신 실패")

# ── CF 파싱 ──────────────────────────────────────────────────
cf = parse_qend(cf_raw)
cf["year"] = cf["date_mapped"].dt.year

for col in ["revenue", "depreciationAndAmortization", "capitalExpenditure",
            "changeInWorkingCapital", "operatingCashFlow", "freeCashFlow",
            "netIncome", "stockBasedCompensation", "amortizationOfIntangibles"]:
    if col in cf.columns:
        cf[col] = pd.to_numeric(cf[col], errors="coerce")

cf_2025 = cf[cf["year"] == TARGET_YEAR].copy()

# ── BS 파싱 & NWC 계산 ────────────────────────────────────────
bs = parse_qend(bs_raw)
bs["year"] = bs["date_mapped"].dt.year

for col in ["currentAssets", "cashAndCashEquivalents",
            "totalCurrentLiabilities", "shortTermDebt"]:
    bs[col] = pd.to_numeric(bs[col], errors="coerce").fillna(0) \
          if col in bs.columns \
          else pd.Series(0.0, index=bs.index)

# 코드와 동일한 NWC 계산식
bs["NWC_calc"] = (
    (bs["currentAssets"] - bs["cashAndCashEquivalents"])
    - (bs["totalCurrentLiabilities"] - bs["shortTermDebt"])
)
bs_2025 = bs[bs["year"] == TARGET_YEAR].copy()

print(f"\n[완료] CF: {len(cf)}분기  (year range: {cf['year'].min()}~{cf['year'].max()})")
print(f"[완료] BS: {len(bs)}분기  (year range: {bs['year'].min()}~{bs['year'].max()})")
print(f"[완료] {TARGET_YEAR}년 CF: {len(cf_2025)}분기  BS: {len(bs_2025)}분기")

[1/2] Cash Flow Statement 수신 중...  (NFLX)
[2/2] Balance Sheet Statement 수신 중...  (NFLX)

[완료] CF: 20분기  (year range: 2021~2025)
[완료] BS: 20분기  (year range: 2021~2025)
[완료] 2025년 CF: 4분기  BS: 4분기


In [6]:
# ════════════════════════════════════════════════════════════
#  [A] Cash Flow Statement — 전체 컬럼 목록
# ════════════════════════════════════════════════════════════

print(f"{'─'*65}")
print(f"  [A] Cash Flow Statement 전체 컬럼 ({len(cf_raw.columns)}개)")
print(f"{'─'*65}")
for i, col in enumerate(cf_raw.columns, 1):
    print(f"  {i:>3}. {col}")

─────────────────────────────────────────────────────────────────
  [A] Cash Flow Statement 전체 컬럼 (40개)
─────────────────────────────────────────────────────────────────
    1. date
    2. symbol
    3. reportedCurrency
    4. cik
    5. fillingDate
    6. acceptedDate
    7. calendarYear
    8. period
    9. netIncome
   10. depreciationAndAmortization
   11. deferredIncomeTax
   12. stockBasedCompensation
   13. changeInWorkingCapital
   14. accountsReceivables
   15. inventory
   16. accountsPayables
   17. otherWorkingCapital
   18. otherNonCashItems
   19. netCashProvidedByOperatingActivities
   20. investmentsInPropertyPlantAndEquipment
   21. acquisitionsNet
   22. purchasesOfInvestments
   23. salesMaturitiesOfInvestments
   24. otherInvestingActivites
   25. netCashUsedForInvestingActivites
   26. debtRepayment
   27. commonStockIssued
   28. commonStockRepurchased
   29. dividendsPaid
   30. otherFinancingActivites
   31. netCashUsedProvidedByFinancingActivities
   32. effect

In [7]:
# ════════════════════════════════════════════════════════════
#  [B] D&A / CAPEX / Working Capital 관련 컬럼 원본값
# ════════════════════════════════════════════════════════════

KEYWORDS = ["deprec", "amort", "capital", "working", "freecash", "operating",
            "stockbased", "netincome", "revenue"]

da_cols = [c for c in cf_raw.columns
           if any(k in c.lower() for k in KEYWORDS)]

print(f"{'─'*65}")
print(f"  [B] D&A·CAPEX·WorkingCapital 관련 컬럼  ({TARGET_YEAR}년 원본값)")
print(f"{'─'*65}")

if not cf_2025.empty and da_cols:
    show_cols = ["date", "date_mapped"] + [c for c in da_cols if c in cf_2025.columns]
    disp = cf_2025[show_cols].copy()
    for c in show_cols[2:]:
        disp[c] = pd.to_numeric(disp[c], errors="coerce")
    display(disp)
else:
    print(f"  {TARGET_YEAR}년 데이터 없음  (CF 보유 기간: {cf['year'].min()}~{cf['year'].max()})")

─────────────────────────────────────────────────────────────────
  [B] D&A·CAPEX·WorkingCapital 관련 컬럼  (2025년 원본값)
─────────────────────────────────────────────────────────────────


,date,date_mapped,netIncome,depreciationAndAmortization,stockBasedCompensation,changeInWorkingCapital,otherWorkingCapital,netCashProvidedByOperatingActivities,operatingCashFlow,capitalExpenditure,freeCashFlow
0,2025-12-31,2025-12-31,2418521000,4850219000,134624000,-252362000,-370252000,2111642000,2111642000,-239335000,1872307000
1,2025-09-30,2025-09-30,2546916000,4090070000,80986000,575750000,436299000,2825174000,2825174000,-164719000,2660455000
2,2025-06-30,2025-06-30,3125413000,3912087000,80862000,-684861000,-695907000,2423258000,2423258000,-155889000,2267369000
3,2025-03-31,2025-03-31,2890351000,3903179000,71977000,-94747000,181679000,2789199000,2789199000,-128277000,2660922000


In [8]:
# ════════════════════════════════════════════════════════════
#  [C] 핵심 항목 요약 — CF Statement + BS NWC 나란히 비교
# ════════════════════════════════════════════════════════════

print(f"{'─'*65}")
print(f"  [C-1] Cash Flow Statement  ({TARGET_YEAR}년)")
print(f"{'─'*65}")

cf_show_cols = {
    "date_mapped":                  "분기말(mapped)",
    "date":                         "FMP 보고일",
    "revenue":                      "Revenue",
    "depreciationAndAmortization":  "D&A (CF)",
    "capitalExpenditure":           "CapEx (CF, 음수=유출)",
    "changeInWorkingCapital":       "ΔWorkingCapital (CF)",
    "stockBasedCompensation":       "SBC",
    "operatingCashFlow":            "Operating CF",
    "freeCashFlow":                 "Free CF",
    "netIncome":                    "Net Income",
}
avail = {k: v for k, v in cf_show_cols.items() if k in cf_2025.columns}

if not cf_2025.empty:
    display(cf_2025[list(avail.keys())].rename(columns=avail))
else:
    print(f"  {TARGET_YEAR}년 CF 데이터 없음")

print()
print(f"{'─'*65}")
print(f"  [C-2] Balance Sheet — NWC 계산  ({TARGET_YEAR}년)")
print(f"  NWC = (currentAssets - cash) - (currentLiab - shortTermDebt)")
print(f"  ※ 코드는 CF의 changeInWorkingCapital 대신 이 값을 사용")
print(f"{'─'*65}")

bs_show_cols = {
    "date_mapped":              "분기말(mapped)",
    "currentAssets":            "CurrentAssets",
    "cashAndCashEquivalents":   "Cash",
    "totalCurrentLiabilities":  "CurrentLiab",
    "shortTermDebt":            "ShortTermDebt",
    "NWC_calc":                 "NWC (코드 계산값)",
}
avail_bs = {k: v for k, v in bs_show_cols.items() if k in bs_2025.columns}

if not bs_2025.empty:
    display(bs_2025[list(avail_bs.keys())].rename(columns=avail_bs))
else:
    print(f"  {TARGET_YEAR}년 BS 데이터 없음")

─────────────────────────────────────────────────────────────────
  [C-1] Cash Flow Statement  (2025년)
─────────────────────────────────────────────────────────────────


,분기말(mapped),FMP 보고일,D&A (CF),"CapEx (CF, 음수=유출)",ΔWorkingCapital (CF),SBC,Operating CF,Free CF,Net Income
0,2025-12-31,2025-12-31,4850219000,-239335000,-252362000,134624000,2111642000,1872307000,2418521000
1,2025-09-30,2025-09-30,4090070000,-164719000,575750000,80986000,2825174000,2660455000,2546916000
2,2025-06-30,2025-06-30,3912087000,-155889000,-684861000,80862000,2423258000,2267369000,3125413000
3,2025-03-31,2025-03-31,3903179000,-128277000,-94747000,71977000,2789199000,2660922000,2890351000



─────────────────────────────────────────────────────────────────
  [C-2] Balance Sheet — NWC 계산  (2025년)
  NWC = (currentAssets - cash) - (currentLiab - shortTermDebt)
  ※ 코드는 CF의 changeInWorkingCapital 대신 이 값을 사용
─────────────────────────────────────────────────────────────────


,분기말(mapped),CurrentAssets,Cash,CurrentLiab,ShortTermDebt,NWC (코드 계산값)
0,2025-12-31,0,9033681000,10980930000,998865000,"-19,015,746,000"
1,2025-09-30,0,9287287000,9731859000,0,"-19,019,146,000"
2,2025-06-30,0,8177405000,8942335000,0,"-17,119,740,000"
3,2025-03-31,0,7199848000,9718519000,1443149000,"-15,475,218,000"


In [9]:
# ════════════════════════════════════════════════════════════
#  [D] CAPEX 부호 검증 — take_abs=True 효과 확인
# ════════════════════════════════════════════════════════════

print(f"{'─'*65}")
print(f"  [D] CAPEX 부호 검증  (코드: take_abs=True 적용)")
print(f"{'─'*65}")
print(f"  {'분기':<12} {'FMP 원본 (음수=유출)':>22} {'abs 적용 후':>18} {'Yahoo 표시 방식'}")
print(f"  {'─'*65}")

if not cf_2025.empty and "capitalExpenditure" in cf_2025.columns:
    for _, row in cf_2025.iterrows():
        raw_val = pd.to_numeric(row.get("capitalExpenditure"), errors="coerce")
        abs_val = abs(raw_val) if pd.notna(raw_val) else float("nan")
        try:
            qtr = row["date_mapped"].strftime("%Y-Q") + str(row["date_mapped"].quarter)
        except:
            qtr = "N/A"
        print(f"  {qtr:<12} {raw_val/1e9:>+18.3f} B  {abs_val/1e9:>14.3f} B  양수(동일값)")
else:
    print(f"  {TARGET_YEAR}년 CAPEX 데이터 없음")

print()
print("  ▶ FMP capitalExpenditure 는 현금 유출이므로 음수로 보고됩니다.")
print("  ▶ 코드의 take_abs=True 가 이를 양수로 변환 → Yahoo 표시값과 일치해야 합니다.")
print("  ▶ 불일치 시: FMP와 Yahoo의 CAPEX 정의 범위 차이 (유지보수 vs 확장 포함 여부) 확인 필요")

─────────────────────────────────────────────────────────────────
  [D] CAPEX 부호 검증  (코드: take_abs=True 적용)
─────────────────────────────────────────────────────────────────
  분기                   FMP 원본 (음수=유출)           abs 적용 후 Yahoo 표시 방식
  ─────────────────────────────────────────────────────────────────
  2025-Q4                  -0.239 B           0.239 B  양수(동일값)
  2025-Q3                  -0.165 B           0.165 B  양수(동일값)
  2025-Q2                  -0.156 B           0.156 B  양수(동일값)
  2025-Q1                  -0.128 B           0.128 B  양수(동일값)

  ▶ FMP capitalExpenditure 는 현금 유출이므로 음수로 보고됩니다.
  ▶ 코드의 take_abs=True 가 이를 양수로 변환 → Yahoo 표시값과 일치해야 합니다.
  ▶ 불일치 시: FMP와 Yahoo의 CAPEX 정의 범위 차이 (유지보수 vs 확장 포함 여부) 확인 필요


In [10]:
# ════════════════════════════════════════════════════════════
#  [E] 최근 12분기 이력 + α·β·γ 비율
# ════════════════════════════════════════════════════════════

print(f"{'─'*65}")
print(f"  [E] D&A / CAPEX / NWC — 최근 12분기 이력 및 Sales 대비 비율")
print(f"{'─'*65}")
print(f"  α = D&A / Revenue   (코드가 추정하는 D&A 계수)")
print(f"  β = |CapEx| / Revenue  (코드가 추정하는 CapEx 계수)")
print(f"  γ = NWC(BS) / Revenue  (코드가 추정하는 NWC 계수)")
print()

# CF 최근 12분기
n = min(12, len(cf))
hist = cf.head(n)[["date_mapped"]].copy()

if "revenue" in cf.columns:
    hist["Revenue"]       = cf["revenue"].head(n).values
if "depreciationAndAmortization" in cf.columns:
    hist["D&A (CF)"]      = cf["depreciationAndAmortization"].head(n).values
if "capitalExpenditure" in cf.columns:
    raw_capex             = pd.to_numeric(cf["capitalExpenditure"].head(n), errors="coerce")
    hist["CapEx(raw)"]    = raw_capex.values
    hist["CapEx(abs)"]    = raw_capex.abs().values
if "changeInWorkingCapital" in cf.columns:
    hist["ΔWC(CF직접값)"] = cf["changeInWorkingCapital"].head(n).values

# BS NWC 머지
bs_nwc = bs[["date_mapped", "NWC_calc"]].head(n)
hist = hist.merge(bs_nwc.rename(columns={"NWC_calc": "NWC(BS계산)"}),
                  on="date_mapped", how="left")

# 비율 계산
if "Revenue" in hist.columns:
    rev = hist["Revenue"].replace(0, float("nan"))
    if "D&A (CF)" in hist.columns:
        hist["α(D&A/Rev)"]    = (hist["D&A (CF)"]   / rev).round(4)
    if "CapEx(abs)" in hist.columns:
        hist["β(CapEx/Rev)"]  = (hist["CapEx(abs)"] / rev).round(4)
    if "NWC(BS계산)" in hist.columns:
        hist["γ(NWC/Rev)"]    = (hist["NWC(BS계산)"] / rev).round(4)

hist = hist.rename(columns={"date_mapped": "분기말"})
display(hist)

─────────────────────────────────────────────────────────────────
  [E] D&A / CAPEX / NWC — 최근 12분기 이력 및 Sales 대비 비율
─────────────────────────────────────────────────────────────────
  α = D&A / Revenue   (코드가 추정하는 D&A 계수)
  β = |CapEx| / Revenue  (코드가 추정하는 CapEx 계수)
  γ = NWC(BS) / Revenue  (코드가 추정하는 NWC 계수)



,분기말,D&A (CF),CapEx(raw),CapEx(abs),ΔWC(CF직접값),NWC(BS계산)
0,2025-12-31,4850219000,-239335000,239335000,-252362000,"-19,015,746,000"
1,2025-09-30,4090070000,-164719000,164719000,575750000,"-19,019,146,000"
2,2025-06-30,3912087000,-155889000,155889000,-684861000,"-17,119,740,000"
3,2025-03-31,3903179000,-128277000,128277000,-94747000,"-15,475,218,000"
4,2024-12-31,4241040000,-158674000,158674000,-70461000,"-16,347,198,000"
5,2024-09-30,3780435000,-126863000,126863000,179579000,"-15,911,094,000"
6,2024-06-30,3850917000,-78287000,78287000,-247227000,"-14,547,888,000"
7,2024-03-31,3758039000,-75714000,75714000,105034000,"-15,117,419,000"
8,2023-12-31,3840646000,-81632000,81632000,59004000,"-15,194,412,000"
9,2023-09-30,3664013000,-103929000,103929000,-75745000,"-14,914,949,000"


In [11]:
# ════════════════════════════════════════════════════════════
#  [F] 계수 중앙값 요약
#  OLS R² < 0.30 또는 표본 < 20개 시 코드가 실제로 사용하는 값
# ════════════════════════════════════════════════════════════

print(f"{'─'*65}")
print(f"  [F] 계수 중앙값 요약  (median ratio fallback 기준값)")
print(f"{'─'*65}")
print()

rows_summary = []
if "α(D&A/Rev)" in hist.columns:
    med = hist["α(D&A/Rev)"].median()
    rows_summary.append({"계수": "α  D&A / Revenue",
                         "median": round(med, 4),
                         "의미": f"매출 1원당 D&A ≈ {med*100:.2f}%"})
if "β(CapEx/Rev)" in hist.columns:
    med = hist["β(CapEx/Rev)"].median()
    rows_summary.append({"계수": "β  CapEx / Revenue",
                         "median": round(med, 4),
                         "의미": f"매출 1원당 CapEx ≈ {med*100:.2f}%"})
if "γ(NWC/Rev)" in hist.columns:
    med = hist["γ(NWC/Rev)"].median()
    rows_summary.append({"계수": "γ  NWC(BS) / Revenue",
                         "median": round(med, 4),
                         "의미": f"매출 1원당 NWC ≈ {med*100:.2f}%"})

if rows_summary:
    display(pd.DataFrame(rows_summary))

print()
print("  ▶ 코드 estimate_ratio_coef() 우선순위:")
print("    1) OLS slope  (R² ≥ 0.30  AND  표본 ≥ 20  AND  slope ≥ 0)")
print("    2) median ratio  (위 조건 미충족 시 → 이 값이 사용됨)")
print()
print("  ▶ 역사적 테이블의 D&A·CapEx·NWC 는 실제 보고값이 아니라")
print("    위 계수 × 해당 분기 Revenue 로 역산한 추정값입니다.")
print()
print("  ▶ Yahoo Finance 불일치 원인 체크리스트:")
print("    □ [C] ΔWC(CF직접값) vs NWC(BS계산) 차이 확인")
print("    □ [D] CapEx abs 변환 후에도 Yahoo 값과 다르면 범위 정의 상이")
print("    □ [E] D&A — FMP CF vs Yahoo IS 라인 차이 (SBC 포함 여부 등)")

─────────────────────────────────────────────────────────────────
  [F] 계수 중앙값 요약  (median ratio fallback 기준값)
─────────────────────────────────────────────────────────────────


  ▶ 코드 estimate_ratio_coef() 우선순위:
    1) OLS slope  (R² ≥ 0.30  AND  표본 ≥ 20  AND  slope ≥ 0)
    2) median ratio  (위 조건 미충족 시 → 이 값이 사용됨)

  ▶ 역사적 테이블의 D&A·CapEx·NWC 는 실제 보고값이 아니라
    위 계수 × 해당 분기 Revenue 로 역산한 추정값입니다.

  ▶ Yahoo Finance 불일치 원인 체크리스트:
    □ [C] ΔWC(CF직접값) vs NWC(BS계산) 차이 확인
    □ [D] CapEx abs 변환 후에도 Yahoo 값과 다르면 범위 정의 상이
    □ [E] D&A — FMP CF vs Yahoo IS 라인 차이 (SBC 포함 여부 등)
